# setup + imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

In [2]:
M1 = np.load("./data/embeddings.npy")
M2 = np.load("./data/subtask1.npy")
M3 = np.load("./data/subtask2.npy")

# data exploration

In [3]:
print(M1.shape, M2.shape, M3.shape)
print(np.linalg.norm(M1[:5], axis=1))

(4000, 384) (4000, 768) (4000, 768)
[0.99999998 1.         1.00000005 1.00000004 1.00000004]


# model (stretch lmao)

In [4]:
def relrepr(M, anchors):
    A = M[anchors]                        
    R = []
    for i in range(len(M)):
        row = A @ M[i]                    
        row = row - row.mean()
        row = row / np.sqrt(np.sum(row ** 2))
        R.append(row)
    return np.array(R)

In [5]:
def bootstrap(Ma, Mb, k, step, rounds):
    aa = list(range(k))
    ab = list(range(k))
    for r in range(rounds):
        sims = relrepr(Ma, aa) @ relrepr(Mb, ab).T  
        abest = sims.argmax(axis=1)                
        bbest = sims.argmax(axis=0)                 
        srt = np.sort(sims, axis=1)
        margin = srt[:, -1] - srt[:, -2]          
        
        pairs = []
        for i in range(len(Ma)):
            j = abest[i]
            if bbest[j] == i and i >= k and j >= k:   
                pairs.append((margin[i], i, j))
        pairs.sort(reverse=True)                  

        aa = list(range(k))
        ab = list(range(k))
        for m, i, j in pairs[:step * (r + 1)]:   
            aa.append(i)
            ab.append(j)

    sims = relrepr(Ma, aa) @ relrepr(Mb, ab).T
    return sims.argmax(axis=1)

# validare

## val s1

In [6]:
fit = list(range(300))

R1 = relrepr(M1, fit)
R2 = relrepr(M2, fit)

pred = []
for i in range(300, 400):
    scores = R2 @ R1[i]          
    pred.append(int(np.argmax(scores)))

nrc = 0
for i in range(300, 400):
  if pred[i-300] == i:            
    nrc += 1
print(nrc/100)

0.87


In [7]:
for step in [25, 50, 100]:
    pred = bootstrap(M1, M2, k=20, step=step, rounds=20)
    print('step', step, 'accuracy on rows 20-399:', np.mean(pred[20:400] == np.arange(20, 400)))

step 25 accuracy on rows 20-399: 0.9105263157894737
step 50 accuracy on rows 20-399: 0.9157894736842105
step 100 accuracy on rows 20-399: 0.9236842105263158


In [8]:
BEST_STEP = 100
BEST_ROUNDS = 20

# subtask1

In [9]:
match2 = bootstrap(M1, M2, k=20, step=BEST_STEP, rounds=BEST_ROUNDS)

# subtask 2

In [10]:
match3 = bootstrap(M1, M3, k=20, step=BEST_STEP, rounds=BEST_ROUNDS)

# submission

In [12]:
N = 4000

subtask1 = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': np.arange(N),        
    'answer': match2
})

subtask2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': np.arange(N),
    'answer': match3
})
submission = pd.concat([subtask1, subtask2], ignore_index=True)
submission.to_csv('submission.csv', index=False)